In [0]:
%pip install -r ../../../requirements.txt
dbutils.library.restartPython()

In [0]:
import pandas as pd
import mlflow
from sklearn import datasets

In [0]:
dbutils.widgets.text("model_name", "mlops_quickstart_dev.iris.iris_model")
dbutils.widgets.text("model_version", "1")
dbutils.widgets.text("deployment_job_id", "")

model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
deployment_job_id = dbutils.widgets.get("deployment_job_id")

In [0]:
# Pull the dataset for running the inference
iris_samples = datasets.load_iris(as_frame=True)
df_samples = pd.DataFrame(data = iris_samples['data'], columns = iris_samples['feature_names'])
df_samples.columns = df_samples.columns.str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
df_samples['species'] = iris_samples.target.astype(int)
df_samples.head()

In [0]:
# REQUIRED: add evaluation dataset and target here
eval_data = df_samples
target = "species"
# REQUIRED: add model type here (e.g. "regressor", "databricks-agent", etc.)
model_type = "classifier"

model_uri = f'models:/{model_name}/{model_version}'
# can also fetch model ID and use that for URI instead as described below

with mlflow.start_run(run_name="evaluation") as run:
  mlflow.models.evaluate(
    model=model_uri,
    data=eval_data,
    targets=target,
    model_type=model_type
  )

## Link the deployment job to the model

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

In [0]:
# deployment_job_id is passed by the job via {{job.id}} task parameter.
# When blank (interactive run), skip linking.
if deployment_job_id:
    client.update_registered_model(model_name, deployment_job_id=deployment_job_id)
    print(f"Linked job '{deployment_job_id}' as the deployment job for model '{model_name}'.")
else:
    print("Not running as a job task — skipping deployment job linking.")